In [ ]:
!pip install ipywidgets
!pip install seaborn
!pip install RPLCD smbus2

In [ ]:
import torch
import cv2
import Jetson.GPIO as GPIO
import time
import ipywidgets
from IPython.display import display

# 載入YOLOv5 模型
print("正在載入模型，請稍候⋯⋯")
model = torch.hub.load('ultralytics/yolov5:v7.0', 'yolov5s', device='cuda:0')

# 43: knife, 76: scissors
TARGET_CLASSES = [43, 76] 

print("模型載入完成！")

In [ ]:
# LCD 設定
try:
    from RPLCD.i2c import CharLCD
    lcd = CharLCD(i2c_expander='PCF8574', address=0x27, port=1, cols=16, rows=2, dotsize=8)
    lcd.clear()
    print("LCD 初始化成功！")
except ImportError:
    print("請執行: !pip install RPLCD smbus2")
    lcd = None
except Exception as e:
    print(f"LCD 初始化失敗: {e}")
    lcd = None

# GPIO 腳位設定
GPIO_BUZZER = 33
GPIO_LED = 7

GPIO.setmode(GPIO.BOARD)
GPIO.setup(GPIO_BUZZER, GPIO.OUT, initial=GPIO.LOW)
GPIO.setup(GPIO_LED, GPIO.OUT, initial=GPIO.LOW)

# SOS 時間單位
UNIT = 0.2

# 警報功能
def alert_signal(duration):
    GPIO.output(GPIO_BUZZER, GPIO.HIGH)
    GPIO.output(GPIO_LED, GPIO.HIGH)
    time.sleep(duration)
    GPIO.output(GPIO_BUZZER, GPIO.LOW)
    GPIO.output(GPIO_LED, GPIO.LOW)

def dot():
    alert_signal(UNIT)
    time.sleep(UNIT)

def dash():
    alert_signal(UNIT * 3)
    time.sleep(UNIT)

def sos():
    # S: ...
    for _ in range(3):
        dot()
    time.sleep(UNIT * 2) 

    # O: ---
    for _ in range(3):
        dash()
    time.sleep(UNIT * 2)

    # S: ...
    for _ in range(3):
        dot()

# main function

# 建立顯示
image_widget = ipywidgets.Image(format='jpeg')
display(image_widget)

# 開啟攝影機
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

print("開始偵測危險")

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("無法讀取攝影機畫面")
            break

        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = model(img_rgb)
        
        danger_detected_now = False

        # 解析結果
        for *xyxy, conf, cls in results.xyxy[0]:
            
            # 判斷是否為危險物品
            if int(cls) in TARGET_CLASSES and conf > 0.1:
                danger_detected_now = True
                
                label = f"{model.names[int(cls)]} {conf:.2f}"
                x1, y1, x2, y2 = map(int, xyxy)
                
                # 將危險物品圈出，並顯示SOS 文字
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 3)
                cv2.putText(frame, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
                cv2.putText(frame, "SOS !!!", (30, 80), cv2.FONT_HERSHEY_SIMPLEX, 2.5, (0, 0, 255), 5)
                
                # 更新Jupyter 畫面
                _, jpeg_frame = cv2.imencode('.jpg', frame)
                image_widget.value = jpeg_frame.tobytes()
                
                # 控制LCD 顯示文字
                if lcd:
                    try:
                        lcd.clear()
                        lcd.cursor_pos = (0, 0)
                        lcd.write_string('Danger Detected!') # 第一行
                        lcd.cursor_pos = (1, 0)
                        lcd.write_string('Status: SOS')      # 第二行
                    except:
                        pass

                # SOS 警報
                print("偵測到危險！ 啟動警報")
                sos()

                # SOS 結束，冷卻3秒
                print("警報結束，冷卻中⋯⋯")
                time.sleep(3)
                
                # 清除 LCD
                if lcd:
                    try:
                        lcd.clear()
                    except:
                        pass
                
                # 清除攝影機緩衝區
                for _ in range(5): 
                    cap.read() 
                
                # 讀取最新畫面並更新
                ret, frame = cap.read()
                if ret:
                    _, jpeg_frame = cv2.imencode('.jpg', frame)
                    image_widget.value = jpeg_frame.tobytes()
                
                # 跳出Detection 迴圈
                break 

        # 如果這幀"沒有"危險，更新畫面
        if not danger_detected_now:
            _, jpeg_frame = cv2.imencode('.jpg', frame)
            image_widget.value = jpeg_frame.tobytes()
        
except KeyboardInterrupt:
    print("程式停止中")

finally:
    cap.release()
    # 關閉硬體
    GPIO.output(GPIO_BUZZER, GPIO.LOW)
    GPIO.output(GPIO_LED, GPIO.LOW)
    GPIO.cleanup()
    # 清除 LCD
    if lcd:
        try:
            lcd.clear()
            lcd.write_string('System Off')
        except:
            pass
    print("資源已釋放")